In [1]:
import pandas as pd
import shutil
from pathlib import Path
import json

In [2]:
carpeta_json = Path("/Users/saitamawick98/Documents/💻/Febrero 2026/TBProfiler_faltan67")

archivos_json = list(carpeta_json.glob("*.json"))
len(archivos_json)

1829

In [3]:
for p in archivos_json:
    print(f"Procesando archivo: {p.stem}")

Procesando archivo: ERR2513872.results
Procesando archivo: SRR1166131.results
Procesando archivo: ERR2515498.results
Procesando archivo: MYCTB171
Procesando archivo: MYC028
Procesando archivo: ERR2510446.results
Procesando archivo: SRR2101427.results
Procesando archivo: ERR1034618.results
Procesando archivo: ERR2516924.results
Procesando archivo: SRR6152742.results
Procesando archivo: ERR2517085.results
Procesando archivo: ERR551289.results
Procesando archivo: ERR2515261.results
Procesando archivo: ERR1034790.results
Procesando archivo: ERR2510564.results
Procesando archivo: ERR1873545.results
Procesando archivo: ERR2513950.results
Procesando archivo: SRR1166163.results
Procesando archivo: ERR551477.results
Procesando archivo: ERR2510365.results
Procesando archivo: ERR551829.results
Procesando archivo: SRR2101304.results
Procesando archivo: SRR2101351.results
Procesando archivo: MYCTB126
Procesando archivo: SRR2100175.results
Procesando archivo: ERR2510414.results
Procesando archivo: S

In [ ]:
import json
import pandas as pd

# 1. Definimos el mapeo de nombres del JSON a tus siglas
TARGET_DRUGS = {
    "rifampicin": "RIF",
    "pyrazinamide": "PZA",
    "isoniazid": "INH",
    "streptomycin": "STR",
    "ethambutol": "EMB"
}

nuevos = []

for p in archivos_json:
    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Información base común
    base_data = {
        "id": data.get("id"),
        "file": p.stem,
        "Linaje": data.get("main_lineage"),
        "Sub_linaje": data.get("sub_lineage"),
        "Clasificacion_Muestra": data.get("drtype"),
        "Spoligotipo.binario": (data.get("spoligotype") or {}).get("binary"),
        "Octal": (data.get("spoligotype") or {}).get("octal"),
        "Familia": (data.get("spoligotype") or {}).get("family"),
        "SIT": (data.get("spoligotype") or {}).get("SIT"),
        "qc.genome_median_depth": (data.get("qc") or {}).get("genome_median_depth"),
    }

    # Aplanamos todas las mutaciones con sus anotaciones en una lista simple
    # para que sea fácil buscarlas después
    todas_las_anotaciones = []
    for v in data.get("dr_variants", []):
        for b in v.get("annotation", []):
            # Combinamos datos de la variante y su anotación
            item = {
                "drug": b.get("drug"),
                "gen": v.get("gene_name"),
                "cambio": v.get("change"),
                "pos": v.get("pos"),
                "depth": v.get("depth"),
                "tipo": b.get("type"),
                "confidencia": b.get("confidence"),
                "comentario": b.get("comment")
            }
            todas_las_anotaciones.append(item)

    # 2. ITERAMOS POR LOS 5 ANTIBIÓTICOS OBJETIVO (OBLIGATORIO)
    for drug_json_name, sigla in TARGET_DRUGS.items():
        
        # Buscamos si este antibiótico tiene mutaciones en el JSON
        matches = [m for m in todas_las_anotaciones if m["drug"] == drug_json_name]

        if matches:
            # Si hay mutación(es), creamos una fila por cada una
            for m in matches:
                fila = base_data.copy()
                fila.update({
                    "Antibiotico_Reporte": sigla,
                    "Resultado": "Resistente/Variante",
                    "Gen": m["gen"],
                    "Cambio": m["cambio"],
                    "Posicion": m["pos"],
                    "Cobertura": m["depth"],
                    "Anotacion.Tipo": m["tipo"],
                    "Anotacion.Confidencia": m["confidencia"],
                    "Anotacion.Comentarios": m["comentario"]
                })
                nuevos.append(fila)
        else:
            # Si NO hay mutación, creamos una fila vacía/sensible para ese antibiótico
            fila = base_data.copy()
            fila.update({
                "Antibiotico_Reporte": sigla,
                "Resultado": "No detectado (Sensible)",
                "Gen": None,
                "Cambio": None,
                "Posicion": None,
                "Cobertura": None,
                "Anotacion.Tipo": None,
                "Anotacion.Confidencia": None,
                "Anotacion.Comentarios": None
            })
            nuevos.append(fila)

df = pd.DataFrame(nuevos)

In [5]:
df

,id,file,Linaje,Sub_linaje,Clasificacion_Muestra,Spoligotipo.binario,Octal,Familia,SIT,qc.genome_median_depth,Antibiotico_Reporte,Resultado,Gen,Cambio,Posicion,Cobertura,Anotacion.Tipo,Anotacion.Confidencia,Anotacion.Comentarios
0,ERR2513872,ERR2513872.results,lineage3,lineage3,Pre-XDR-TB,■■■□□□□■■■■■■■■■■■■■■□□□□□□□□□□□□□■■■■■■■■■,703777700003771,CAS1-Delhi,142,53.0,RIF,Resistente/Variante,rpoB,p.Ser450Leu,761155.0,72.0,drug_resistance,Assoc w R,
1,ERR2513872,ERR2513872.results,lineage3,lineage3,Pre-XDR-TB,■■■□□□□■■■■■■■■■■■■■■□□□□□□□□□□□□□■■■■■■■■■,703777700003771,CAS1-Delhi,142,53.0,PZA,No detectado (Sensible),None,None,NaN,NaN,None,None,None
2,ERR2513872,ERR2513872.results,lineage3,lineage3,Pre-XDR-TB,■■■□□□□■■■■■■■■■■■■■■□□□□□□□□□□□□□■■■■■■■■■,703777700003771,CAS1-Delhi,142,53.0,INH,Resistente/Variante,katG,p.Asn138His,2155700.0,39.0,drug_resistance,Uncertain significance,Mutation from literature
3,ERR2513872,ERR2513872.results,lineage3,lineage3,Pre-XDR-TB,■■■□□□□■■■■■■■■■■■■■■□□□□□□□□□□□□□■■■■■■■■■,703777700003771,CAS1-Delhi,142,53.0,INH,Resistente/Variante,katG,p.Asn138His,2155700.0,39.0,who_confidence,Uncertain significance,
4,ERR2513872,ERR2513872.results,lineage3,lineage3,Pre-XDR-TB,■■■□□□□■■■■■■■■■■■■■■□□□□□□□□□□□□□■■■■■■■■■,703777700003771,CAS1-Delhi,142,53.0,STR,No detectado (Sensible),None,None,NaN,NaN,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10423,ERR552358,ERR552358.results,lineage2,lineage2.2.1,MDR-TB,□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□■■■■■■■■■,000000000003771,Beijing,1,79.0,PZA,No detectado (Sensible),None,None,NaN,NaN,None,None,None
10424,ERR552358,ERR552358.results,lineage2,lineage2.2.1,MDR-TB,□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□■■■■■■■■■,000000000003771,Beijing,1,79.0,INH,Resistente/Variante,katG,p.Ser315Thr,2155168.0,92.0,drug_resistance,Assoc w R,High-level resistance
10425,ERR552358,ERR552358.results,lineage2,lineage2.2.1,MDR-TB,□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□■■■■■■■■■,000000000003771,Beijing,1,79.0,STR,Resistente/Variante,rpsL,p.Lys43Arg,781687.0,91.0,drug_resistance,Assoc w R,
10426,ERR552358,ERR552358.results,lineage2,lineage2.2.1,MDR-TB,□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□□■■■■■■■■■,000000000003771,Beijing,1,79.0,STR,Resistente/Variante,rrs,n.1401A>G,1473246.0,141.0,who_confidence,Uncertain significance,


In [6]:
df.to_csv(carpeta_json/"TBProfiler_67_samples.csv", index=False)